# 0. Build the surface-footprint sensitivity cache

This is the only expensive notebook. It applies the corrected surface calculation to prespecified filled ellipses and annuli, plus two legacy controls. The depth-following cache is not read or changed.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
for path in (ANALYSIS_ROOT, HERE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
import footprint_tools as ft
sns.set_theme(style="whitegrid", context="notebook")
palette = {"AE":"#c44e52", "CE":"#4c72b0"}

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
eddies, _ = tilt.load_tilt_tables(paths)
specs = ft.default_footprints()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8), constrained_layout=True)
for _, row in specs.iterrows():
    y = {"Filled":2, "Annulus":1, "Legacy":0}[row.kind]
    ax.plot([row.inner_frac, row.frac], [y, y], lw=9, solid_capstyle="butt", alpha=.75,
            label=row.kind if row.footprint == specs[specs.kind.eq(row.kind)].footprint.iloc[0] else None)
ax.set(yticks=[0,1,2], yticklabels=["Legacy", "Annulus", "Corrected filled"], xlabel="Linear ellipse scale (frac)", title="Prespecified footprint experiment")
ax.legend(frameon=False, ncol=3); plt.show()

In [ ]:
data = ft.build_cache(eddies, grid, specs)
destination = ft.cache_path()
destination.parent.mkdir(parents=True, exist_ok=True)
data.to_parquet(destination, index=False)
print(f"Saved {len(data):,} rows to {destination}")

In [ ]:
counts = data.groupby("footprint").agg(snapshots=("Day","size"), eddies=("Eddy","nunique"), median_cells=("PV_footprint_n","median")).reset_index()
fig, axes = plt.subplots(1,2, figsize=(12,4.2), constrained_layout=True)
sns.barplot(data=counts, x="footprint", y="snapshots", color="0.35", ax=axes[0])
sns.barplot(data=counts, x="footprint", y="median_cells", color="tab:green", ax=axes[1])
for ax in axes: ax.tick_params(axis="x", rotation=45)
axes[0].set(title="Rows retained", xlabel=""); axes[1].set(title="Median ocean cells per footprint", xlabel="")
plt.show()